# Tome 1, Chapter 7: Sums of Squares (Python companion)

Exercises first, solutions at the end. Computer-science threads: **Cornacchia's algorithm** (writing $p=x^2+y^2$ with one
Euclidean algorithm), **Monte Carlo and lattice counting** for $\pi$, and a **randomised four-square algorithm**
(Rabin–Shallit style).

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from sympy import isprime, randprime, factorint, divisors, sqrt_mod

## Exercises

**P7.1 (★) Who is a sum of two squares?** Compute, by brute force, the set of $n\le 10^4$ that are sums of two squares, and check Theorem 7.3 (the criterion on primes $\equiv 3 \pmod 4$) against it. Do the same for three squares and the $4^a(8b+7)$ criterion.

In [ ]:
# Your code here


**P7.2 (★★) Jacobi's formula, numerically.** Compute $r_2(n)$ by brute force for $n \le 5000$ and compare with $4\sum_{d\mid n}\chi_4(d)$. (The proof comes in Chapter 10: here we only gather evidence.)

In [ ]:
# Your code here


**P7.3 (★★) CS application: Cornacchia's algorithm.** Let $p\equiv1\pmod4$ and $a^2\equiv-1\pmod p$ with $p/2<a<p$. Run Euclid's algorithm on $(p,a)$ and stop at the first remainder $r<\sqrt p$: then $p=r^2+s^2$ where $s$ is the next remainder. Implement it, test it on 1000 random 200-bit primes, and compare with Thue's lemma (which only says *some* small $x,y$ exist).

*How to think about it:* Thue's lemma is a pigeonhole argument, non-constructive. Euclid's algorithm on $(p,a)$ produces, along the way, all the "best" small pairs $(x,y)$ with $ax\equiv\pm y$, so it is the natural way to *find* the pair Thue promises.

In [ ]:
# Your code here


**P7.4 (★) Gauss' circle, and $\pi$.** Plot $N(R)-\pi R^2$ for $R\le 300$ together with the bound of Theorem 7.5 and the curve $\pm R^{1/2}$. Then estimate $\pi$ in two ways: lattice counting $N(R)/R^2$, and Monte Carlo with random points in the square. Which converges faster, and why?

In [ ]:
# Your code here


**P7.5 (★★★) CS application: four squares, fast.** Brute force over $x_1,x_2,x_3$ costs $O(n^{3/2})$. Faster: pick random $x_1,x_2$ with $n-x_1^2-x_2^2=p$ a prime $\equiv1\pmod 4$ (or twice one), then split $p$ with Cornacchia. Implement this for odd $n$ around $10^{40}$.

*How to think about it:* the bottleneck in the brute force is searching for the last two squares. Theorem 7.2 plus Cornacchia makes "is $m$ a sum of two squares, and which?" instant when $m$ is prime, and primes are dense enough (density $\approx 1/\ln m$) that a few hundred random tries suffice.

In [ ]:
# Your code here


**P7.6 (★★) Zagier's involution in action.** For $p=97$, list the set $S$ of Exercise 7.7, check that $f$ is an involution with a single fixed point, and then follow the orbit 'alternately apply $f$ and $g$' starting from the fixed point of $f$: it ends at a fixed point of $g$, i.e. a decomposition $p=x^2+(2y)^2$. This turns Zagier's existence proof into an algorithm (though not a fast one).

In [ ]:
# Your code here


---
## Solutions

### P7.1
The criterion is checked by factoring; brute force is the ground truth.

In [ ]:
N = 10_000
two = set()
for x in range(int(math.isqrt(N)) + 1):
    for y in range(x, int(math.isqrt(N - x * x)) + 1):
        two.add(x * x + y * y)

def two_sq_criterion(n):
    return all(e % 2 == 0 for q, e in factorint(n).items() if q % 4 == 3)

assert all((n in two) == two_sq_criterion(n) for n in range(1, N + 1))
print("two-square criterion OK up to", N, "; density", len([n for n in two if 0 < n <= N]) / N)

sq = [k * k for k in range(int(math.isqrt(N)) + 1)]
three = set()
for i, a in enumerate(sq):
    for j in range(i, len(sq)):
        for k in range(j, len(sq)):
            s = a + sq[j] + sq[k]
            if s > N:
                break
            three.add(s)

def bad_form(n):
    while n % 4 == 0:
        n //= 4
    return n % 8 == 7

assert all((n in three) != bad_form(n) for n in range(1, N + 1))
print("three-square criterion OK up to", N)

### P7.2

In [ ]:
M = 5000
r2 = [0] * (M + 1)
for x in range(-71, 72):
    for y in range(-71, 72):
        if x * x + y * y <= M:
            r2[x * x + y * y] += 1

chi4 = lambda d: 0 if d % 2 == 0 else (1 if d % 4 == 1 else -1)
assert all(r2[n] == 4 * sum(chi4(d) for d in divisors(n)) for n in range(1, M + 1))
print("Jacobi's formula holds for n <=", M, "; e.g. r2(25) =", r2[25], ", r2(65) =", r2[65])
print("average of r2 up to M:", sum(r2[1:]) / M, "vs pi =", math.pi)

### P7.3
Why it works (sketch): the remainders of Euclid on $(p,a)$ satisfy $r_k\equiv \pm t_k a$ with the Bézout coefficients $t_k$, and $|t_k|$ grows while $r_k$ shrinks with $r_k|t_{k+1}|\le p$. At the crossover both are below $\sqrt p$, which is exactly Thue's situation, so $r^2+t^2=p$; one shows $t$ equals the next remainder (Brillhart's improvement, 1972).

In [ ]:
def cornacchia(p):
    a = sqrt_mod(-1, p)
    if a < p // 2:
        a = p - a
    r0, r1 = p, a
    s = math.isqrt(p)
    while r1 > s:
        r0, r1 = r1, r0 % r1
    x = r1
    y = math.isqrt(p - x * x)
    assert x * x + y * y == p
    return x, y

print(cornacchia(1013), cornacchia(2029))
random.seed(1)
for _ in range(1000):
    p = randprime(2**199, 2**200)
    while p % 4 != 1:
        p = randprime(2**199, 2**200)
    x, y = cornacchia(p)
print("1000 random 200-bit primes split; last:", x, y)

### P7.4
Lattice counting has error $O(R)$ on $N\approx\pi R^2$ points, i.e. relative error $O(1/R)=O(1/\sqrt{\text{points}})$ at worst and much better in practice (the true error is about $R^{1/2}$, relative $\approx \text{points}^{-3/4}$). Monte Carlo has relative error $\approx 1/\sqrt{\text{points}}$: random sampling is the slow one here, because the lattice is a very regular sample. (This is the idea behind *quasi-Monte Carlo* methods.)

In [ ]:
Rs = np.arange(1, 301)
def N_of(R):
    return sum(2 * math.isqrt(R * R - x * x) + 1 for x in range(-R, R + 1))
err = np.array([N_of(R) - math.pi * R * R for R in Rs])
plt.figure(figsize=(7, 3.5))
plt.plot(Rs, err, lw=0.8, label="N(R) - pi R^2")
plt.plot(Rs, 2 * math.sqrt(2) * math.pi * Rs + 2 * math.pi, "r--", label="bound of Thm 7.5")
plt.plot(Rs, -(2 * math.sqrt(2) * math.pi * Rs + 2 * math.pi), "r--")
plt.plot(Rs, Rs ** 0.5, "k:", label="+-sqrt(R)"); plt.plot(Rs, -Rs ** 0.5, "k:")
plt.ylim(-150, 150); plt.legend(); plt.show()

R = 300
print("lattice:", N_of(R) / R**2, " error", abs(N_of(R) / R**2 - math.pi))
rng = np.random.default_rng(0)
pts = rng.uniform(-1, 1, size=(N_of(R), 2))
mc = 4 * np.mean((pts ** 2).sum(axis=1) <= 1)
print("Monte Carlo with as many points:", mc, " error", abs(mc - math.pi))

### P7.5
*How to choose the parities.* We want $m=n-x_1^2-x_2^2$ to be odd and $\equiv1\pmod4$, so that a prime $m$ splits by Theorem 7.2. Squares are $0\pmod4$ for even numbers and $1\pmod4$ for odd ones. If $n\equiv1\pmod4$, take $x_1,x_2$ both even: $m\equiv1$. If $n\equiv3\pmod4$, take both odd: $m\equiv3-2=1$. Among such $m$ of size $\approx n$, primes have density about $2/\ln n$, hence the few dozen tries below.

In [ ]:
def four_squares(n, rng=random.Random(0)):
    assert n % 2 == 1
    par = 0 if n % 4 == 1 else 1            # parity of x1, x2 so that m = 1 mod 4
    s = math.isqrt(n)
    tries = 0
    while True:
        tries += 1
        x1 = rng.randrange(par, s // 2, 2) if s // 2 > par else par
        x2 = rng.randrange(par, s // 2, 2) if s // 2 > par else par
        m = n - x1 * x1 - x2 * x2
        if m > 0 and m % 4 == 1 and isprime(m):
            x3, x4 = cornacchia(m)
            assert x1**2 + x2**2 + x3**2 + x4**2 == n
            return (x1, x2, x3, x4), tries

for n in [10**40 + 1, 10**40 + 3, 2**131 - 1]:
    sol, tries = four_squares(n)
    print(n, "=", " + ".join(f"{x}^2" for x in sol), f"  ({tries} tries)")

### P7.6

In [ ]:
p = 97
S = [(x, y, z) for x in range(1, math.isqrt(p) + 1) for y in range(1, p) for z in range(1, p)
     if x * x + 4 * y * z == p]

def f(t):
    x, y, z = t
    if x < y - z:
        return (x + 2 * z, z, y - x - z)
    if x < 2 * y:
        return (2 * y - x, y, x - y + z)
    return (x - 2 * y, x - y + z, y)

g = lambda t: (t[0], t[2], t[1])
assert all(f(t) in S and f(f(t)) == t for t in S)
fixed = [t for t in S if f(t) == t]
print("|S| =", len(S), " fixed points of f:", fixed)

t, path = fixed[0], [fixed[0]]
while g(t) != t:
    t = g(t); path.append(t)
    if f(t) == t: break
    t = f(t); path.append(t)
x, y, _ = t
print("path length", len(path), "->", t, ":", p, "=", f"{x}^2 + {2*y}^2")
assert x * x + (2 * y) ** 2 == p